# Q3 · T4 데이터·학습 확장

**학습 영상 128개 · 검증 영상 32개 · 추가 2,000 업데이트**를 실행합니다.
첫 검증(16/4 영상, 300 업데이트)의 best.pt에서 모델 가중치만 가져옵니다.
데이터 선택이 달라지므로 optimizer·난수·step은 새로 시작합니다.
같은 검증 영상 32개에서 시작 가중치, 학습 후 모델, copy-last를 비교합니다.
실제 행동 라벨은 사용하지 않습니다. 예측은 과거 32장 → 미래 32장입니다.


## 1. 확장 실험 설정
첫 검증 결과는 별도 폴더에 보존하고 새 실행 폴더에 저장합니다.
공식 에피소드별 train/dev 구분을 유지합니다. 선택 영상 수를 늘리면 부분집합이 달라지므로
이전 검증 4개의 평균과 새 검증 32개의 평균을 직접 비교하지 않습니다.
같은 이름의 latest.pt가 있으면 이 확장 실험을 재개합니다.


In [ ]:
import time
REPO_URL = "https://github.com/seungjoolee24/krafton-q3-video-worldmodel.git"
GIT_REF = "main"
PROFILE = "pilot"
RUN_NAME = "video-only-t4-expanded-v1"
DRIVE_ROOT = "/content/drive/MyDrive/krafton-q3-video-worldmodel"
DATA_ZIP = DRIVE_ROOT + "/track3-kit.zip"
DATA_SHA256 = "dbd9da58b386782533913da2c44b088a6c25c6119ba6efbe680dae56bf4422ee"
INIT_CHECKPOINT = DRIVE_ROOT + "/runs/video-only-t4-first-v1/best.pt"
RESUME_CHECKPOINT = ""
MAX_STEPS = 2000
BATCH_SIZE = 4
EXPANDED_STARTED = time.time()
print({"train_videos": 128, "dev_videos": 32, "new_updates": MAX_STEPS,
       "initialize_weights_from": INIT_CHECKPOINT, "actions_used": False})


## 2. 코드와 GPU 환경
현재 Git 커밋을 출력합니다. 재현할 때 `GIT_REF`에 이 SHA를 사용하세요.
Colab에 설치된 CUDA Torch를 유지하고 나머지 의존성만 설치합니다.

In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path
import torch
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요."
version = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
assert version >= (2, 3), f"Torch >= 2.3 required, found {torch.__version__}"
assert PROFILE in {"pilot", "full"} and BATCH_SIZE >= 1 and MAX_STEPS >= 0
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME, "RUN_NAME은 폴더 이름 하나로 지정하세요."
CODE = Path("/content/q3-video-code")
if not (CODE / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(CODE)], check=True)
else:
    remote = subprocess.check_output(["git", "-C", str(CODE), "remote", "get-url", "origin"], text=True).strip()
    assert remote == REPO_URL, "Existing checkout has a different repository."
    dirty = subprocess.check_output(["git", "-C", str(CODE), "status", "--porcelain"], text=True)
    assert not dirty.strip(), "Source changes exist; preserve them before changing revision."
subprocess.run(["git", "-C", str(CODE), "fetch", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(CODE), "checkout", "--detach", "FETCH_HEAD"], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(CODE), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(CODE / "requirements-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(CODE), "--no-deps"], check=True)
os.chdir(CODE)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
print({"commit": COMMIT, "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0)})
subprocess.run([sys.executable, "scripts/check_project.py"], check=True)


## 3. Drive 연결과 원본 데이터
Google Drive 접근은 본인 계정으로 승인합니다.
데이터 ZIP을 런타임 로컬 디스크로 복사한 뒤 압축을 풉니다.
학습 중 Drive에서 수천 개의 파일을 반복해서 읽지 않습니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = Path(DRIVE_ROOT)
DRIVE.mkdir(parents=True, exist_ok=True)
source_zip = Path(DATA_ZIP)
LOCAL_DATA = Path("/content/q3-data")
LOCAL_DATA.mkdir(exist_ok=True)
local_zip = LOCAL_DATA / "track3-kit.zip"
from importlib.util import spec_from_file_location, module_from_spec
specification = spec_from_file_location("unpack_kit", CODE / "scripts/unpack_kit.py")
unpack_module = module_from_spec(specification)
specification.loader.exec_module(unpack_module)
unpack_module.stage_archive(source_zip, local_zip, DATA_SHA256)
KIT = unpack_module.unpack(local_zip, LOCAL_DATA / "unpacked")
CACHE = Path(f"/content/q3-cache-{PROFILE}")
LOCAL_RUN = Path(f"/content/q3-runs/{RUN_NAME}")
DRIVE_RUN = DRIVE / "runs" / RUN_NAME
config = json.loads((CODE / "configs" / f"{PROFILE}.json").read_text())
config["batch_size"] = BATCH_SIZE
config["evaluate_at_start"] = True
if MAX_STEPS:
    config["total_steps"] = MAX_STEPS
LOCAL_RUN.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = LOCAL_RUN / "config.json"
CONFIG_PATH.write_text(json.dumps(config, indent=2))
print({"kit": str(KIT), "cache": str(CACHE), "drive_results": str(DRIVE_RUN), "config": config})


## 4. 짧은 동작 검증
작은 합성 입력의 gradient·배치 독립성·미래 정보 차단·재개를 검사합니다.
GPU smoke는 두 번의 작은 합성 업데이트만 수행하며 학습 가중치를 저장하지 않습니다.

In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)
subprocess.run([sys.executable, "-m", "video_wam.cli", "smoke", "--kit", str(KIT), "--device", "cuda"], check=True)


## 5. 영상 특징 캐시
행동 sidecar는 읽지 않습니다. 원본 RGB를 제공된 고정 인코더로 변환합니다.
Pilot는 약 3GB, 전체는 약 35GB의 캐시 공간이 필요합니다.
중단된 같은 런타임에서 다시 실행하면 검증된 에피소드 캐시를 재사용합니다.
런타임이 삭제되면 원본 ZIP에서 다시 생성합니다.

In [ ]:
subprocess.run([sys.executable, "-m", "video_wam.cli", "prepare",
                "--kit", str(KIT), "--cache", str(CACHE), "--config", str(CONFIG_PATH),
                "--encode-batch", "64", "--device", "cuda"], check=True)
index = json.loads((CACHE / "index.json").read_text())
print({"train_episodes": len(index["train_ids"]), "dev_episodes": len(index["dev_ids"]),
       "actions_used": index["actions_used"], "cache_fingerprint": index["fingerprint"]})


## 6. 2,000回の追加学習
最初に、引き継いだモデルを新しい検証32本で評価します。
その後 1 → 4 → 8 → 16 → 32 の予測長で追加学習します。
新しいデータに合わせて optimizer と KL ウォームアップを初期化します。
この実験自身の latest.pt からの再開時は optimizer・step・乱数も戻します。
100回ごとに保存、200回ごとに32本の検証を行い、結果をDriveに保管します。


In [ ]:
resume = Path(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else DRIVE_RUN / "latest.pt"
command = [sys.executable, "-m", "video_wam.cli", "train", "--kit", str(KIT),
           "--cache", str(CACHE), "--config", str(CONFIG_PATH), "--out", str(LOCAL_RUN),
           "--persist", str(DRIVE_RUN), "--device", "cuda"]
if RESUME_CHECKPOINT:
    assert resume.is_file(), f"Resume checkpoint not found: {resume}"
if resume.is_file():
    command.extend(["--resume", str(resume)])
    print("Resume:", resume)
else:
    initial_checkpoint = Path(INIT_CHECKPOINT)
    assert initial_checkpoint.is_file(), f"Initial weights not found: {initial_checkpoint}"
    command.extend(["--init-from", str(initial_checkpoint)])
    print("Initialize model weights only:", initial_checkpoint)
if MAX_STEPS:
    command.extend(["--max-steps", str(MAX_STEPS)])
subprocess.run(command, check=True)


## 7. 同じ32本で学習前後を比較
最終モデルと、検証誤差が最小だった best.pt を区別して示します。
RGB MSE は全画素、物体領域 MSE は色を使った補助指標です。
比較動画は最小検証誤差のモデルです。公式スコアや力の復元精度ではありません。


In [ ]:
from IPython.display import display, HTML, Video
completion_path = LOCAL_RUN / "completion.json"
if not completion_path.exists():
    completion_path = DRIVE_RUN / "completion.json"
completion = json.loads(completion_path.read_text())

def report_for(step):
    relative = Path("validation") / f"step_{step:06d}"
    folder = LOCAL_RUN / relative
    if not (folder / "metrics.json").exists():
        folder = DRIVE_RUN / relative
    return folder, json.loads((folder / "metrics.json").read_text())

_, initial = report_for(0)
final_folder, final = report_for(completion["step"])
best_checkpoint = torch.load(DRIVE_RUN / "best.pt", map_location="cpu", weights_only=True)
best_folder, best = report_for(best_checkpoint["step"])
rows = [
    ("학습 전 (첫 검증 가중치)", initial["mse"]["mean"], initial["foreground_mse_heuristic"]["mean"]),
    (f"최종 모델 (추가 {completion['step']:,}회)", final["mse"]["mean"], final["foreground_mse_heuristic"]["mean"]),
    (f"최소 검증 오차 모델 ({best_checkpoint['step']:,}회)", best["mse"]["mean"], best["foreground_mse_heuristic"]["mean"]),
    ("copy-last 기준선", final["copy_last_mse"]["mean"], final["copy_last_foreground_mse_heuristic"]["mean"]),
]
table_rows = "".join(f"<tr><td>{name}</td><td>{rgb:.8f}</td><td>{fg:.6f}</td></tr>"
                     for name, rgb, fg in rows)
display(HTML("<table><thead><tr><th>검증 영상 32개 · 미래 32프레임</th>"
             "<th>RGB MSE ↓</th><th>물체 영역 MSE ↓</th></tr></thead><tbody>"
             + table_rows + "</tbody></table>"))
print(f"학습 전 대비 최종 RGB 오차 감소: {(1 - final['mse']['mean'] / initial['mse']['mean']) * 100:.2f}%")
print(f"copy-last 대비 최종 RGB 오차 감소: {(1 - final['mse']['mean'] / final['copy_last_mse']['mean']) * 100:.2f}%")
print(f"완료 업데이트: {completion['step']:,} / {completion['requested_steps']:,}; 종료: {completion['reason']}")
print(f"학습·검증 시간: {completion['wall_seconds'] / 60:.2f}분; 전체 실행: {(time.time() - EXPANDED_STARTED) / 60:.2f}분")
print(f"학습 GPU 최대 할당 메모리: {completion['gpu_peak_memory_gb']:.3f} GB; 실제 행동 라벨 사용: False")
print("시점별 최종 RGB MSE:", {key: final["mse"][key] for key in ["h1", "h8", "h16", "h32"]})
print("저장 위치:", DRIVE_RUN)
for video in sorted(best_folder.glob("*.mp4")):
    display(Video(str(video), embed=True, width=680))
